In [ ]:
# Borrar caché de modelos gemma
!rm -rf /root/.cache/huggingface/hub/models--google--gemma*

In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.6 MB/s eta 0:00:00


In [ ]:
import torch
import bitsandbytes as bnb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

1. Definición del modelo que vamos a utilizar

In [ ]:
model_checkpoint = "google/gemma-2-2b-it"
n_labels = 2

2. Configuración de Cuatización a 4-bits

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Cargando tokenizador y modelo base Gemma")

Cargando tokenizador y modelo base Gemma


3. Tokenizador

In [ ]:
"""from google.colab import userdata
from huggingface_hub import login, hf_hub_download
from transformers import AutoTokenizer

# 1. Configuración
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# 2. Descarga FORZADA del archivo problemático
# Esto descarga el archivo grande primero de forma aislada
print("Descargando tokenizer.json...")
try:
    hf_hub_download(repo_id=model_checkpoint, filename="tokenizer.json", token=hf_token)
    print("¡Descarga de tokenizer.json completada!")
except Exception as e:
    print(f"Error en la descarga: {e}")

# 3. Carga del Tokenizer (ahora como el archivo ya está en caché local,
# AutoTokenizer lo leerá al instante sin intentar descargarlo de nuevo)
print("Cargando tokenizer desde caché...")
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, token=hf_token)
print("¡Éxito!")"""

'from google.colab import userdata\nfrom huggingface_hub import login, hf_hub_download\nfrom transformers import AutoTokenizer\n\n# 1. Configuración\nhf_token = userdata.get(\'HF_TOKEN\')\nlogin(token=hf_token)\n\n# 2. Descarga FORZADA del archivo problemático\n# Esto descarga el archivo grande primero de forma aislada\nprint("Descargando tokenizer.json...")\ntry:\n    hf_hub_download(repo_id=model_checkpoint, filename="tokenizer.json", token=hf_token)\n    print("¡Descarga de tokenizer.json completada!")\nexcept Exception as e:\n    print(f"Error en la descarga: {e}")\n\n# 3. Carga del Tokenizer (ahora como el archivo ya está en caché local,\n# AutoTokenizer lo leerá al instante sin intentar descargarlo de nuevo)\nprint("Cargando tokenizer desde caché...")\ntokenizer = AutoTokenizer.from_pretrained(model_checkpoint, token=hf_token)\nprint("¡Éxito!")'

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

4. Inicialización del Modelo Base Cuantizado

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=n_labels,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Gemma2ForSequenceClassification LOAD REPORT from: google/gemma-2-2b-it
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


5. Buscador dinámico de capas para LoRA

In [ ]:
def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:
        lora_module_names.remove('lm_head')
    if 'score' in lora_module_names:
        lora_module_names.remove('score')
    return list(lora_module_names)

modules = find_all_linear_names(model)
print(f"Módulos objetivo encontrados para LoRA: {modules}")

Módulos objetivo encontrados para LoRA: ['k_proj', 'up_proj', 'down_proj', 'o_proj', 'gate_proj', 'q_proj', 'v_proj']


6. Configuración de LoRA

In [ ]:
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=modules,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    modules_to_save=["score"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 20,771,328 || all params: 2,635,117,824 || trainable%: 0.7883


7. Preparación del dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ================================
# 1. Carga y preparación de datos
# ================================
import pandas as pd
from sklearn.model_selection import train_test_split
# from datasets import Dataset

print("Cargando el dataset maestro de entrenamiento...")
# Cargamos el TRAIN_MASTER que contiene el 80% de los datos (el test fijo ya está guardado aparte)
df_train_master = pd.read_csv("/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1_master.csv")
test_df = pd.read_csv('/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv')

# Limpiamos el índice extra (si existe) y renombramos la columna
if "Unnamed: 0" in df_train_master.columns:
    df_train_master = df_train_master.drop(columns=["Unnamed: 0"])
df_train_master = df_train_master.rename(columns={"label_task_3_1_merged": "label"})
df_train_master["label"] = df_train_master["label"].astype(int)

if "Unnamed: 0" in test_df.columns:
    test_df = test_df.drop(columns=["Unnamed: 0"])
test_df = test_df.rename(columns={"label_task_3_1_merged": "label"})
test_df["label"] = test_df["label"].astype(int)

# División dinámica: 90% Train, 10% Validation (extraído solo del bloque maestro)
# Mantenemos random_state para que la validación sea estable entre pruebas del mismo modelo
train_df, val_df = train_test_split(df_train_master, test_size=0.10, stratify=df_train_master["label"], random_state=42)

print("Distribución fichero de entrenamiento (Train):")
print(train_df['label'].value_counts())

print("\nDistribución fichero de validación (Valid):")
print(val_df['label'].value_counts())

print("\nDistribución fichero de test (estático):")
print(test_df['label'].value_counts())

# train_dataset = Dataset.from_pandas(train_df)
# eval_dataset = Dataset.from_pandas(val_df)

Cargando el dataset maestro de entrenamiento...
Distribución fichero de entrenamiento (Train):
label
0    940
1    865
Name: count, dtype: int64

Distribución fichero de validación (Valid):
label
0    105
1     96
Name: count, dtype: int64

Distribución fichero de test (estático):
label
0    261
1    241
Name: count, dtype: int64


In [ ]:
def tokenize_data(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(val_df)

train_dataset.reset_format()
valid_dataset.reset_format()

columns_train = train_dataset.column_names
columns_valid = valid_dataset.column_names
columna_etiqueta = "label" if "label" in columns_train else "labels"

if columna_etiqueta in columns_train: columns_train.remove(columna_etiqueta)
if columna_etiqueta in columns_valid: columns_valid.remove(columna_etiqueta)

encoded_train_dataset = train_dataset.map(tokenize_data, batched=True, remove_columns=columns_train)
encoded_valid_dataset = valid_dataset.map(tokenize_data, batched=True, remove_columns=columns_valid)


Map:   0%|          | 0/1805 [00:00<?, ? examples/s]

Map:   0%|          | 0/201 [00:00<?, ? examples/s]

8. Hiperparámetros del entrenamiento

In [ ]:
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/Gemma_2B_QLoRA',
    num_train_epochs=5,
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,
    eval_strategy='steps',
    eval_steps=0.2,              # Evaluamos cada 20%
    save_strategy='steps',
    save_steps=0.2,              # OBLIGATORIO: Mismo valor que eval_steps
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    weight_decay=0.01,
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    logging_strategy='steps',
    logging_steps=0.2            # Para ver los logs al mismo tiempo
)

9. Entrenamiento

In [ ]:
import sklearn as sk
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, f1_score


In [ ]:
# Función para realizar distintas métricas en ejecución

def compute_metrics(eval_pred):

  ##############
  ## predictions son logits, que son tuplas de la forma [valor1, valor2]
  ## Por ejemplo [-1.5606991,  1.6122842] significa que ha predicho eso para un documento
  ## Eso es lo que pasa a la última capa del transformer (softmax si es binario)
  ## Por eso se utiliza el índice del valor máximo de la tupla, para decir que esa es la clase que predice

  ## label_ids = [0, 1, 1, 0, 1]  # Etiquetas reales
  ## predictions = [
  ##  [0.8, 0.2],  # Predicciones para la primera instancia
  ##  [0.3, 0.7],  # Predicciones para la segunda instancia
  ##  [0.1, 0.9],  # Predicciones para la tercera instancia
  ##  [0.9, 0.1],  # Predicciones para la cuarta instancia
  ##  [0.4, 0.6],  # Predicciones para la quinta instancia
  ##           ]

  ##############

  labels = eval_pred.label_ids
  preds = eval_pred.predictions.argmax(-1)

  # Compute precision, recall, F1-score, and support
  precision, recall, f1, _ = sk.metrics.precision_recall_fscore_support(labels, preds, average="macro")

  # Calculate F1-score for the minority class (label = 1)
  f1_minoritaria= f1_score(labels, preds, pos_label=1)

  # Calculate F1-score for the majority class (label = 0)
  f1_mayoritaria = f1_score(labels, preds, pos_label=0)

  # Calculate accuracy
  acc = sk.metrics.accuracy_score(labels, preds)

  # Calculate Area Under the Curve (AUC)
  AUC = roc_auc_score(labels, preds)

  # Calculate Precision-Recall Area Under the Curve (AUC)
  PREC_REC = average_precision_score(labels, preds)

  return {
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall,
      'AUC': AUC,
      'f1_minoritaria': f1_minoritaria,
      'f1_mayoritaria': f1_mayoritaria,
      'PREC_REC': PREC_REC
  }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics, # ¡Asegúrate de tener esta función cargada en memoria!
    train_dataset=encoded_train_dataset,
    eval_dataset=encoded_valid_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("🚀 Iniciando entrenamiento de Gemma-2-2B...")
trainer.train()

🚀 Iniciando entrenamiento de Gemma-2-2B...


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Auc,F1 Minoritaria,F1 Mayoritaria,Prec Rec
113,3.447878,0.627610,0.666667,0.663972,0.680738,0.672024,0.672024,0.694064,0.633880,0.588662
226,1.535378,0.733610,0.666667,0.665474,0.665920,0.665327,0.665327,0.645503,0.685446,0.590908
339,0.541207,2.054110,0.741294,0.740979,0.747091,0.744345,0.744345,0.750000,0.731959,0.655400
452,0.091722,2.739497,0.736318,0.735033,0.736314,0.734673,0.734673,0.716578,0.753488,0.658129
565,0.003710,2.928592,0.736318,0.735033,0.736314,0.734673,0.734673,0.716578,0.753488,0.658129


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=565, training_loss=1.1239790960750748, metrics={'train_runtime': 1233.1831, 'train_samples_per_second': 7.318, 'train_steps_per_second': 0.458, 'total_flos': 1.41763405529088e+16, 'train_loss': 1.1239790960750748, 'epoch': 5.0})

# Resultados contra fichero de Test

In [1]:
!pip install -U torchao accelerate transformers

In [13]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, AutoModelForSequenceClassification
from peft import PeftModel

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuración

In [16]:
MODEL_ID_BASE = "google/gemma-2-2b-it" # Cambiar según el modelo
DIR_MODELO_QLORA = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/Gemma_2B_QLoRA/checkpoint-339"

# Ruta al archivo de test estático
CSV_TEST_TEXT = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv"
CSV_SALIDA = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/predicciones_gemma_test.csv"


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Carga de datos de test

In [5]:
print("Cargando el dataset de test fijo...")
test_df = pd.read_csv(CSV_TEST_TEXT)

if "Unnamed: 0" in test_df.columns:
    test_df = test_df.drop(columns=["Unnamed: 0"])
test_df = test_df.rename(columns={"label_task_3_1_merged": "label"})
test_df["label"] = test_df["label"].astype(int)

# Guardamos la verdad absoluta
y_true = test_df["label"].tolist()

# Función para generar el prompt de evaluación (sin la respuesta)
def generate_test_prompt(data_point):
    return f"""
Clasifica el siguiente texto extraído de un vídeo de redes sociales en una de estas dos categorías: 'Misógino' o 'No misógino'. Devuelve ÚNICAMENTE la etiqueta correspondiente.
text: {data_point["text"]}
label: """.strip()

Cargando el dataset de test fijo...


## 3. Carga del modelo y tokenizador (PEFT/QLoRA)

In [11]:
print(f"Cargando Tokenizador de {MODEL_ID_BASE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID_BASE)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Cargando Modelo Base (SequenceClassification) en 16-bits...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID_BASE,
    num_labels=2,            # Fundamental: Le decimos que es para 2 clases
    device_map="auto",
    torch_dtype=torch.bfloat16
)
base_model.config.pad_token_id = tokenizer.pad_token_id

print(f"Aplicando pesos QLoRA (incluyendo la capa de 'score') desde {DIR_MODELO_QLORA}...")
# Esto cargará la cabeza clasificadora gracias a tu "modules_to_save=['score']"
model = PeftModel.from_pretrained(base_model, DIR_MODELO_QLORA)
model.eval()

Cargando Tokenizador de google/gemma-2-2b-it...
Cargando Modelo Base (SequenceClassification) en 16-bits...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

[transformers] Gemma2ForSequenceClassification LOAD REPORT from: google/gemma-2-2b-it
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Aplicando pesos QLoRA (incluyendo la capa de 'score') desde /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/Gemma_2B_QLoRA/checkpoint-339...


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): Gemma2ForSequenceClassification(
      (model): Gemma2Model(
        (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 2304, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2304, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2304, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector

## 4. Inferencia y parseo de las respuestas

In [14]:
print("Iniciando inferencia (extracción de probabilidades)...")
predicciones_csv = []
y_pred = []

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    # Usamos el texto puro, tal cual lo hiciste en el entrenamiento
    texto = str(row["text"])
    id_video = row["id_EXIST"]

    # Tokenizamos
    inputs = tokenizer(texto, return_tensors="pt", padding="max_length", truncation=True, max_length=128).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

        # Obtenemos los logits y aplicamos Softmax para sacar porcentajes (0 a 1)
        logits = outputs.logits
        probabilidades = F.softmax(logits, dim=-1)[0]

        # probabilidad de la clase 1 (Misógino)
        prob_misogino = probabilidades[1].item()

    # Decisión binaria (Umbral estándar > 0.5)
    pred_binaria = 1 if prob_misogino > 0.5 else 0
    y_pred.append(pred_binaria)

    # Guardamos los datos para el Ensemble
    predicciones_csv.append({
        "id_EXIST": id_video,
        "prob_misogino": prob_misogino,
        "prediccion_binaria": pred_binaria,
        "label_real": row["label"]
    })

Iniciando inferencia (extracción de probabilidades)...


100%|██████████| 502/502 [00:40<00:00, 12.28it/s]


## 5. Evaluación y métricas

In [15]:
print("\n" + "="*50)
print(f"🏆 RESULTADOS REALES EN TEST FIJO: Gemma 2B (Sequence Classification)")
print("="*50)

f1 = f1_score(y_true, y_pred, average="macro")
acc = accuracy_score(y_true, y_pred)

print(f"\nF1-Score (Macro): {f1:.4f}")
print(f"Accuracy: {acc:.4f}")

print("\nMatriz de Confusión:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["No Misógino", "Misógino"]))


🏆 RESULTADOS REALES EN TEST FIJO: Gemma 2B (Sequence Classification)

F1-Score (Macro): 0.6866
Accuracy: 0.6873

Matriz de Confusión:
[[161 100]
 [ 57 184]]

Classification Report:
              precision    recall  f1-score   support

 No Misógino       0.74      0.62      0.67       261
    Misógino       0.65      0.76      0.70       241

    accuracy                           0.69       502
   macro avg       0.69      0.69      0.69       502
weighted avg       0.70      0.69      0.69       502



## 6. Guardar CSV para ensemble

In [17]:
df_salida = pd.DataFrame(predicciones_csv)
df_salida.to_csv(CSV_SALIDA, index=False)
print(f"\n✅ ¡CSV para el Ensemble guardado correctamente en: {CSV_SALIDA}!")


✅ ¡CSV para el Ensemble guardado correctamente en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/predicciones_gemma_test.csv!
